In [21]:
import os
import json
import time
import random
import requests
from pathlib import Path
from datetime import date
import pandas as pd
from io import StringIO
from datetime import datetime

class Crawler():
  def __init__(self):
    self.req = requests.Session()
    self.url = "https://mops.twse.com.tw/mops/web/ajax_t164sb03"

    self.headers = self.req.headers
    self.headers["User-Agent"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36(KHTML, like Gecko) Chrome/76.0.3809.132 Safari/537.36"

  def __get_data(self, stock_no, year, season):
    res = self.req.post(self.url, {"encodeURIComponent":1,
                                   "step":1,
                                   "firstin":1,
                                   "off":1,
                                   "TYPEK":"all",
                                   "isnew": "false",
                                   "co_id": stock_no,
                                   "year": year,
                                   "season": season})
    res.encoding = "utf-8"
    return res

  def __solve_data(self, stock_no, year, season):
    data = self.__get_data(stock_no, year, season)
    if data.text.__contains__("查無所需資料"):
      return None
    else:
      html_df = pd.read_html(StringIO(data.text))
      data_form = html_df[1] if len(html_df) == 2 else html_df[2]
      field = []
      field.append("Season")
      df = {}
      df["Season"] = "{}/{}".format(year+1911,season)
      for n in range(0,len(data_form)):
        df_row = data_form.loc[n] # 第n列
        field.append(df_row.iloc[0])
        df[df_row.iloc[0]] = int(df_row.iloc[1]) if not pd.isnull(df_row.iloc[1]) else None # 每一列數據
      result = {
        "field": field,
        "data": df
      }
    return result

  def __save_file(self, path, stock_data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as file:
      file.write(json.dumps(
        stock_data,
        indent=3,
        ensure_ascii=False
      ))

  def Spider(self, stock_no, year, season, path):
    stock_data = self.__solve_data(stock_no, year, season)
    if stock_data == None:
      print("{}/{}/{}_No Data".format(stock_no, year, season))
      pass
    else:
      self.__save_file(path, stock_data)

In [101]:
with open("./Listed_stock_info_list.json","r",encoding="utf-8") as f:
  stock_info_list = json.load(f)

for stock_info in stock_info_list["stock"]:
  stock_no = stock_info["stockNo"]
  stock_name = stock_info["stockName"]
  stock_industry = stock_info["stockIndustry"]
  stock_date = stock_info["stockDate"] # 股票上市日期

  for Y in range(102, 114):
    ED_S = 1 if Y == 113 else 4
    for S in range(1, ED_S+1):
      PATH = "./stock_data/{2} {3}/balance_sheet/{0}_{1}.json".format(Y+1911,S,stock_no,stock_name)
      # 判斷路徑是否存在
      if os.path.exists(PATH) == True:
        continue
      else:
        try:
          Crawler().Spider(stock_no, Y, S, PATH)
        except:
          print("{}/{}/{}_Error".format(stock_no,Y,S))
          pass
        time.sleep(random.randint(100,300)*0.01)

102_1
102_2
102_3
102_4
103_1
103_2
103_3
103_4
104_1
104_2
104_3
104_4
105_1
105_2
105_3
105_4
106_1
106_2
106_3
106_4
107_1
107_2
107_3
107_4
108_1
108_2
108_3
108_4
109_1
109_2
109_3
109_4
110_1
110_2
110_3
110_4
111_1
111_2
111_3
111_4
112_1
112_2
112_3
112_4
113_1
102_1
102_2
102_3
102_4
103_1
103_2
103_3
103_4
104_1
104_2
104_3
104_4
105_1
105_2
105_3


KeyboardInterrupt: 

In [24]:
stock_no = 1203
stock_name = "味王"
YEAR = 106
SEASON = 4
PATH = "./stock_data/{2} {3}/balance_sheet/{0}_{1:02}.json".format(YEAR,SEASON,stock_no,stock_name)
Crawler().Spider(stock_no, YEAR, SEASON, PATH)
print("{}/{}/{}_Error {}".format(stock_no,YEAR,SEASON,datetime.now()))
# data = Crawler().solve_data(stock_no, YEAR, SEASON)
# print(len(data))

1203/106/4_Error 2024-05-25 22:01:15.240434


In [ ]:
with open("./stock_data/1101 台泥/balance_sheet/112_01.json", "r", encoding="utf-8") as test:
 a = json.load(test)
for b in a["data"].keys():
 print(b)

Season
流動資產
現金及約當現金
透過損益按公允價值衡量之金融資產－流動
透過其他綜合損益按公允價值衡量之金融資產－流動
按攤銷後成本衡量之金融資產－流動
應收票據淨額
應收帳款淨額
應收帳款－關係人淨額
其他應收款淨額
其他應收款－關係人淨額
存貨
預付款項
其他流動資產
流動資產合計
非流動資產
透過損益按公允價值衡量之金融資產－非流動
透過其他綜合損益按公允價值衡量之金融資產－非流動
按攤銷後成本衡量之金融資產－非流動
採用權益法之投資
不動產、廠房及設備
使用權資產
投資性不動產淨額
無形資產
其他非流動資產
非流動資產合計
資產總額
流動負債
短期借款
應付短期票券
透過損益按公允價值衡量之金融負債－流動
合約負債－流動
應付帳款
其他應付款
其他應付款項－關係人
本期所得稅負債
租賃負債－流動
其他流動負債
流動負債合計
非流動負債
應付公司債
長期借款
遞延所得稅負債
租賃負債－非流動
其他非流動負債
非流動負債合計
負債總額
歸屬於母公司業主之權益
股本
普通股股本
特別股股本
股本合計
資本公積
資本公積合計
保留盈餘
保留盈餘合計
其他權益
其他權益合計
庫藏股票
歸屬於母公司業主之權益合計
非控制權益
權益總額
負債及權益總計
預收股款（權益項下）之約當發行股數（單位：股）
母公司暨子公司所持有之母公司庫藏股股數（單位：股）


In [ ]:
data_form = pd.read_html(data.text)[1]
h = data_form.loc[0][1]
field = []
field.append("Season")
df = {}
df["Season"] = "112/1"
for n in range(0,len(data_form)):
  df_row = data_form.loc[n] # 第n列
  field.append(df_row[0])
  df[df_row[0]] = int(df_row[1]) if pd.isnull(df_row[1]) == False else None # 每一列數據
result = {
  "field": field,
  "data": df
}
print(field)
print(df)
print(result)
Y=112
S=1
stock_no=1101
stock_name = "台泥"
with open("./stock_data/{2} {3}/balance_sheet/{0}_{1:02}.json".format(Y,S,stock_no,stock_name)", "w", encoding="utf-8") as b_file:
  b_file.write(json.dumps(
    result,
    indent=3,
    ensure_ascii=False
  ))

['Season', '流動資產', '現金及約當現金', '透過損益按公允價值衡量之金融資產－流動', '備供出售金融資產－流動淨額', '應收票據淨額', '應收帳款淨額', '應收帳款－關係人淨額', '其他應收款－關係人淨額', '存貨', '預付款項', '其他流動資產', '流動資產合計', '非流動資產', '備供出售金融資產－非流動淨額', '以成本衡量之金融資產－非流動淨額', '採用權益法之投資淨額', '不動產、廠房及設備', '投資性不動產淨額', '無形資產', '其他非流動資產', '非流動資產合計', '資產總額', '流動負債', '短期借款', '應付短期票券', '應付帳款', '應付帳款－關係人', '其他應付款', '當期所得稅負債', '其他流動負債', '流動負債合計', '非流動負債', '應付公司債', '長期借款', '遞延所得稅負債', '其他非流動負債', '非流動負債合計', '負債總額', '歸屬於母公司業主之權益', '股本', '普通股股本', '股本合計', '資本公積', '資本公積合計', '保留盈餘', '保留盈餘合計', '其他權益', '國外營運機構財務報表換算之兌換差額', '備供出售金融資產未實現損益', '現金流量避險中屬有效避險部分之避險工具利益（損失）', '其他權益合計', '歸屬於母公司業主之權益合計', '非控制權益', '權益總額', '預收股款（權益項下）之約當發行股數（單位：股）', '母公司暨子公司所持有之母公司庫藏股股數（單位：股）']
{'Season': '112/1', '流動資產': None, '現金及約當現金': 18994984, '透過損益按公允價值衡量之金融資產－流動': 271687, '備供出售金融資產－流動淨額': 15066125, '應收票據淨額': 8919088, '應收帳款淨額': 9489955, '應收帳款－關係人淨額': 236032, '其他應收款－關係人淨額': 1820536, '存貨': 10148150, '預付款項': 4568191, '其他流動資產': 7731607, '流動資產合計': 77246355, '非流動資產': None, '備供出售金融資產－非流動淨額': 4775276, '以成本衡量之金融資

C:\Users\cyten\AppData\Local\Temp\ipykernel_50096\4226339776.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  data_form = pd.read_html(data.text)[1]
C:\Users\cyten\AppData\Local\Temp\ipykernel_50096\4226339776.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  h = data_form.loc[0][1]
C:\Users\cyten\AppData\Local\Temp\ipykernel_50096\4226339776.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  field.append(df_row[0])
C:\Users\cyten\AppData\Local\Temp\ipykernel_50096\4226339776.py:10: Futur

FileNotFoundError: [Errno 2] No such file or directory: './stock_data/1101 台泥/balance_sheet/112_01.json'

In [ ]:
with open("./Listed_stock_info_list.json","r",encoding="utf-8") as f:
  stock_info_list = json.load(f)

for stock_info in stock_info_list["stock"]:
  stock_no = stock_info["stockNo"]
  stock_name = stock_info["stockName"]
  stock_industry = stock_info["stockIndustry"]
  stock_date = stock_info["stockDate"] # 股票上市日期

  # 證交所最早的資料(民國99年)
  start_year = int(stock_date[:4]) if int(stock_date[:4]) > 2010 else 2010
  start_month = int(stock_date.split("/")[1])

  # 今天日期
  today = date.today()
  end_year = today.year
  end_month = today.month
  for Y in range(start_year,end_year+1):
    ST_M = start_month if Y == start_year & int(stock_date[:4]) >= 2010 else 1
    ED_M = end_month if Y == end_year else 12
    for M in range(ST_M, ED_M+1):
      D = "01"
      DATE = "{}{:02}{}".format(Y,M,D)
      PATH = "./stockData/{2} {3}/'dateInfo'/{0}_{1:02}.json".format(Y,M,stock_no,stock_name)
      # 判斷路徑是否存在
      if os.path.exists(PATH) == True:
        continue
      else:
        try:
          print(PATH)
          Crawler().save_file(DATE,stock_no,PATH)
        except:
          pass
        time.sleep(random.randint(3,5))

stockData/1103 嘉泥/'dateInfo'/2024_05.json
stockData/1104 環泥/'dateInfo'/2010_01.json
stockData/1104 環泥/'dateInfo'/2010_02.json
stockData/1104 環泥/'dateInfo'/2010_03.json
stockData/1104 環泥/'dateInfo'/2010_04.json
stockData/1104 環泥/'dateInfo'/2010_05.json
stockData/1104 環泥/'dateInfo'/2010_06.json
stockData/1104 環泥/'dateInfo'/2010_07.json
stockData/1104 環泥/'dateInfo'/2010_08.json
stockData/1104 環泥/'dateInfo'/2010_09.json
stockData/1104 環泥/'dateInfo'/2010_10.json
stockData/1104 環泥/'dateInfo'/2010_11.json
stockData/1104 環泥/'dateInfo'/2010_12.json
stockData/1104 環泥/'dateInfo'/2011_01.json
stockData/1104 環泥/'dateInfo'/2011_02.json
stockData/1104 環泥/'dateInfo'/2011_03.json
stockData/1104 環泥/'dateInfo'/2011_04.json
stockData/1104 環泥/'dateInfo'/2011_05.json
stockData/1104 環泥/'dateInfo'/2011_06.json
stockData/1104 環泥/'dateInfo'/2011_07.json
stockData/1104 環泥/'dateInfo'/2011_08.json
stockData/1104 環泥/'dateInfo'/2011_09.json
stockData/1104 環泥/'dateInfo'/2011_10.json
stockData/1104 環泥/'dateInfo'/2011_